## Feature Umap

In [ ]:
import umap
import os
import pandas as pd
import h5py
import openslide
import lmdb
import pickle
import numpy as np

import torch

from datasets.data_utils import imfrombytes
from vis_utils import *

In [ ]:
def get_imgs_panda(name):
    files = pd.read_csv("/XXX/panda/train.csv")
    center_type = int(files[files['image_id'] == name]['data_provider'] == 'Karolinska')
    patch = h5py.File('/XXX/panda/level0_256/patches/' + name + '.h5', "r")
    features = []
    coords = patch['coords']
    mask = openslide.OpenSlide(
        os.path.join("/XXX/panda/train_label_masks", name + '_mask.tiff'))
    mask_data = mask.read_region((0, 0), 0, mask.level_dimensions[0])
    label = get_label_new(coords, mask_data,center_type)

    #  imgs
    env = lmdb.open("/XXX/panda/224_80_jpg.lmdb", subdir=False, readonly=True, lock=False, readahead=False, meminit=False, map_size=100*(1024**3))
    with env.begin(write=False) as txn:
        PN_dict = pickle.loads(txn.get(b'__pn__'))
    imgs_id = [str(name)+'-'+str(i) for i in range(PN_dict[name])]

    imgs = torch.empty((len(imgs_id), 3, 224, 224),memory_format=torch.channels_last)
    imgs = torch.empty((len(imgs_id), 3,224, 224))

    with env.begin(write=False, buffers=True) as txn:
        for i,key_str in enumerate(imgs_id):
            imgs[i] = torch.from_numpy(imfrombytes(pickle.loads(txn.get(key_str.encode('ascii')).tobytes())).transpose(2, 0, 1))

    mean=torch.tensor([0.485, 0.456, 0.406]).view((1,3,1,1))* 255
    std=torch.tensor([0.229, 0.224, 0.225]).view((1,3,1,1))* 255

    imgs.sub_(mean).div_(std)

    print(imgs.shape)
    print(coords.shape)
    return imgs,coords,label

In [ ]:
## anno
_f = 'ff97fa212a451432b685bdad1c3c89df'  # 024 7
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
n_neighbors = 5
min_dist = 0.1

In [ ]:
imgs,coords,label = get_imgs_panda(_f)

In [ ]:
# load abmilx_best checkpoint
model_abx = load_model("config/e2e/r50_panda.yaml",
                      "/XXX/fold_0_model_best.pt",device)

In [ ]:
uni_feat = torch.load('/XXX/panda/uni/pt_files/' + _f + '.pt')
r50_feat = torch.load('/XXX/panda/r50/pt_files/' + _f + '.pt')
with torch.inference_mode():
    abx_feat = model_abx.forward_encoder_batch(imgs,1024)

In [ ]:
#R50
embedding_r50 = umap.UMAP(n_neighbors=n_neighbors,
                      min_dist=min_dist,
                      metric='correlation').fit_transform(r50_feat)
y = label if label is not None else np.array([1 for i in range(len(embedding_r50))])
feat_vis_r50 = plot(embedding_r50, y, draw_legend=False)

In [ ]:
#UNI
embedding_uni = umap.UMAP(n_neighbors=n_neighbors,
                      min_dist=min_dist,
                      metric='correlation').fit_transform(uni_feat)
y = label if label is not None else np.array([1 for i in range(len(embedding_uni))])
feat_vis_uni = plot(embedding_uni, y, draw_legend=False)

In [ ]:
# ABX
embedding_abx = umap.UMAP(n_neighbors=n_neighbors,
                      min_dist=min_dist,
                      metric='correlation').fit_transform(abx_feat.cpu().detach())
y = label if label is not None else np.array([1 for i in range(len(embedding_abx))])
feat_vis_abx = plot(embedding_abx, y, draw_legend=False)